In [ ]:
annotator = 0
image_ids = []
for i in range(2):
    for j in range(10):
        image_ids.append(str(i)+str(j))
root_paths = ["/data/D-Persona/models/DPersona1_NPC_20250114-232424results",
              "/data/D-Persona/models/DPersona2_NPC_20250115-050904results",
              "/data/D-Persona/models/cm_global01_2_NPC_20250122-103651results",
              "/data/D-Persona/models/cm_pixel01_2_NPC_20250122-122047results",
              "/data/D-Persona/models/prob_unet01_2_NPC_20250122-065829results",
              "/data/D-Persona/models/TAB-232522results",
              "/data/D-Persona/models/pionono01_NPC_20250115-001034results",
              "/data/D-Persona/models/pionono_prob01_2_NPC_20250118-012413results",
              "/data/D-Persona/models/pionono_prob01_2_NPC_20250118-012413results_prior"
              ]
method_names = ["DPersona1",
              "DPersona2",
              "cm_global01_2",
              "cm_pixel01_2",
              "prob_unet01_2",
              "TAB",
              "pionono01",
              "pionono_prob",
              "pionono_prob_prior"
              ]

for root_path, method_name in zip(root_paths, method_names):
    print(f'getting distribution of {method_name}')
    get_distribution(root_path, method_name)
if method_name not in root_path:
    raise Exception("method_name not in root_path")



In [ ]:
get_distribution("/data/D-Persona/models/pionono_prob01_2_NPC_20250118-012413results_prior", "pionono_prob_prior")

In [ ]:

import numpy as np
from tqdm import tqdm
import nibabel as nib
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np
from lib.metrics_set import  distance_one as distance
import random

def get_data(path):
    nii_file = nib.load(path)
    data = nii_file.get_fdata()
    #print(data.shape)
    if len(data.shape) == 4:
        data = data.reshape(-1, 128, 128)
    return data

def compute_similarity(slice_id, image_data_1, image_data_2, image_data_3, image_data_4):
    slice_1 = image_data_1[slice_id].astype(np.int32)
    slice_2 = image_data_2[slice_id].astype(np.int32)
    slice_3 = image_data_3[slice_id].astype(np.int32)
    slice_4 = image_data_4[slice_id].astype(np.int32)
    temo_list = [slice_1, slice_2, slice_3, slice_4]
    random.shuffle(temo_list)
    slice_1, slice_2, slice_3, slice_4 = temo_list
    #np.random.shuffle([slice_1, slice_2, slice_3, slice_4])
    #print(slice_1[64,64], slice_2[64,64], slice_3[64,64], slice_4[64,64])
    distance_1_3 = distance(slice_1.flatten(), slice_3.flatten())
    distance_1_2 = distance(slice_1.flatten(), slice_2.flatten())
    distance_4_3 = distance(slice_4.flatten(), slice_3.flatten())
    distance_4_2 = distance(slice_4.flatten(), slice_2.flatten())
    return np.array([distance_1_3+distance_1_2, distance_4_3+distance_4_2])

def get_distribution(root_path, method_name):
    all_slices_similarity = []
    for image_id in tqdm(image_ids):
        image_path_1 = f'{root_path}/{image_id}_pred_s1.nii.gz'
        image_path_2 = f'{root_path}/{image_id}_pred_s2.nii.gz'
        image_path_3 = f'{root_path}/{image_id}_pred_s3.nii.gz'
        image_path_4 = f'{root_path}/{image_id}_pred_s4.nii.gz'
        image_data_1 = get_data(image_path_1)
        image_data_2 = get_data(image_path_2)
        image_data_3 = get_data(image_path_3)
        image_data_4 = get_data(image_path_4)
        num_slices = image_data_1.shape[0]
        temp_similar_matrices = Parallel(n_jobs=-1)(
            delayed(compute_similarity)(one_slice_id, image_data_1, image_data_2, image_data_3, image_data_4) for one_slice_id in tqdm(range(num_slices))
        )
        all_slices_similarity.append(temp_similar_matrices)
    all_slices_similarity = np.array(all_slices_similarity, dtype=object)
    #print(all_slices_similarity.shape)
    
    np.save(f'{method_name}_rand.npy', all_slices_similarity)
    
annotator = 0
image_ids = []
for i in range(2):
    for j in range(10):
        image_ids.append(str(i)+str(j))
root_paths = [
            "/mnt/nas/share2/home/liuke/edata/DPersona1_NPC_20250114-232424results",
              "/mnt/nas/share2/home/liuke/edata/DPersona2_NPC_20250115-050904results",
              "/mnt/nas/share2/home/liuke/edata/cm_global01_2_NPC_20250122-103651results",
              "/mnt/nas/share2/home/liuke/edata/cm_pixel01_2_NPC_20250122-122047results",
              "/mnt/nas/share2/home/liuke/edata/prob_unet01_2_NPC_20250122-065829results",
              "/mnt/nas/share2/home/liuke/edata/TAB-232522results",
              "/mnt/nas/share2/home/liuke/edata/pionono01_NPC_20250115-001034results",
              "/mnt/nas/share2/home/liuke/edata/pionono_prob01_2_NPC_20250118-012413results",
              "/data/D-Persona/models/pionono_prob01_2_NPC_20250118-012413results_prior"
              ]
method_names = ["DPersona1",
              "DPersona2",
              "cm_global01_2",
              "cm_pixel01_2",
              "prob_unet01_2",
              "TAB",
              "pionono01",
              "pionono_prob",
              "pionono_prob_prior"
              ]

for root_path, method_name in zip(root_paths, method_names):
    print(f'getting distribution of {method_name}')
    get_distribution(root_path, method_name)
if method_name not in root_path:
    raise Exception("method_name not in root_path")



In [ ]:
for one_method in method_names:
    one_np = np.load(f'vis/sim/{one_method}.npy')
    print(one_np.shape)
    

In [15]:

import numpy as np
from tqdm import tqdm
import nibabel as nib
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np
from lib.metrics_set import  distance_one as distance

def get_data(path):
    nii_file = nib.load(path)
    data = nii_file.get_fdata()
    #print(data.shape)
    if len(data.shape) == 4:
        data = data.reshape(-1, 128, 128)
    return data

# def compute_similarity(slice_id, image_data_1, image_data_2, image_data_3, image_data_4):
#     slice_1 = image_data_1[slice_id].astype(np.int32)
#     slice_2 = image_data_2[slice_id].astype(np.int32)
#     slice_3 = image_data_3[slice_id].astype(np.int32)
#     slice_4 = image_data_4[slice_id].astype(np.int32)
#     #print(slice_1[64,64], slice_2[64,64], slice_3[64,64], slice_4[64,64])
#     distance_1_3 = distance(slice_1.flatten(), slice_3.flatten())
#     distance_1_2 = distance(slice_1.flatten(), slice_2.flatten())
#     distance_4_3 = distance(slice_4.flatten(), slice_3.flatten())
#     distance_4_2 = distance(slice_4.flatten(), slice_2.flatten())
#     return np.array([distance_1_3+distance_1_2, distance_4_3+distance_4_2])

def compute_similarity(slice_id, image_data_1, image_data_2, image_data_3, image_data_4):
    slice_1 = image_data_1[slice_id].astype(np.int32)
    slice_2 = image_data_2[slice_id].astype(np.int32)
    slice_3 = image_data_3[slice_id].astype(np.int32)
    slice_4 = image_data_4[slice_id].astype(np.int32)
    temo_list = [slice_1, slice_2, slice_3, slice_4]
    random.shuffle(temo_list)
    slice_1, slice_2, slice_3, slice_4 = temo_list
    #np.random.shuffle([slice_1, slice_2, slice_3, slice_4])
    #print(slice_1[64,64], slice_2[64,64], slice_3[64,64], slice_4[64,64])
    distance_1_3 = distance(slice_1.flatten(), slice_3.flatten())
    distance_1_2 = distance(slice_1.flatten(), slice_2.flatten())
    distance_4_3 = distance(slice_4.flatten(), slice_3.flatten())
    distance_4_2 = distance(slice_4.flatten(), slice_2.flatten())
    return np.array([distance_1_3+distance_1_2, distance_4_3+distance_4_2])

def get_label_distribution(root_path, method_name):
    all_slices_similarity = []
    for image_id in tqdm(image_ids):
        image_path_1 = f'{root_path}/{image_id}_label_a1.nii.gz'
        image_path_2 = f'{root_path}/{image_id}_label_a2.nii.gz'
        image_path_3 = f'{root_path}/{image_id}_label_a3.nii.gz'
        image_path_4 = f'{root_path}/{image_id}_label_a4.nii.gz'
        image_data_1 = get_data(image_path_1)
        image_data_2 = get_data(image_path_2)
        image_data_3 = get_data(image_path_3)
        image_data_4 = get_data(image_path_4)
        num_slices = image_data_1.shape[0]
        temp_similar_matrices = Parallel(n_jobs=-1)(
            delayed(compute_similarity)(one_slice_id, image_data_1, image_data_2, image_data_3, image_data_4) for one_slice_id in tqdm(range(num_slices))
        )
        all_slices_similarity.append(temp_similar_matrices)
    all_slices_similarity = np.array(all_slices_similarity, dtype=object)
    #print(all_slices_similarity.shape)
    
    np.save(f'vis/sim/Gold_rand.npy', all_slices_similarity)
    


In [ ]:
get_label_distribution("/mnt/nas/share2/home/liuke/prj/D-Persona-main/D-Persona/models/pionono_prob01_2_NPC_20250118-012413results_prior", "pionono_prob_prior")

In [ ]:

import numpy as np
from tqdm import tqdm
import nibabel as nib
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np
from lib.metrics_set import  distance_one as distance

def get_data(path):
    nii_file = nib.load(path)
    data = nii_file.get_fdata()
    print(data.shape)
    if len(data.shape) == 4:
        data = data.reshape(-1, 128, 128)
    return data

def compute_similarity(slice_id, image_data_1, image_data_2, image_data_3, image_data_4):
    slice_1 = image_data_1[slice_id].astype(np.int32)
    slice_2 = image_data_2[slice_id].astype(np.int32)
    slice_3 = image_data_3[slice_id].astype(np.int32)
    slice_4 = image_data_4[slice_id].astype(np.int32)
    #print(slice_1[64,64], slice_2[64,64], slice_3[64,64], slice_4[64,64])
    distance_1_3 = distance(slice_1.flatten(), slice_3.flatten())
    distance_1_2 = distance(slice_1.flatten(), slice_2.flatten())
    distance_4_3 = distance(slice_4.flatten(), slice_3.flatten())
    distance_4_2 = distance(slice_4.flatten(), slice_2.flatten())
    return np.array([distance_1_3+distance_1_2, distance_4_3+distance_4_2])

def get_distribution(root_path, method_name):
    all_slices_similarity = []
    for image_id in tqdm(image_ids):
        image_path_1 = f'{root_path}/{image_id}_pred_s1.nii.gz'
        image_path_2 = f'{root_path}/{image_id}_pred_s2.nii.gz'
        image_path_3 = f'{root_path}/{image_id}_pred_s3.nii.gz'
        image_path_4 = f'{root_path}/{image_id}_pred_s4.nii.gz'
        image_data_1 = get_data(image_path_1)
        image_data_2 = get_data(image_path_2)
        image_data_3 = get_data(image_path_3)
        image_data_4 = get_data(image_path_4)
        num_slices = image_data_1.shape[0]
        temp_similar_matrices = Parallel(n_jobs=-1)(
            delayed(compute_similarity)(one_slice_id, image_data_1, image_data_2, image_data_3, image_data_4) for one_slice_id in tqdm(range(num_slices))
        )
        all_slices_similarity.append(temp_similar_matrices)
    all_slices_similarity = np.array(all_slices_similarity, dtype=object)
    #print(all_slices_similarity.shape)
    
    np.save(f'{method_name}_new.npy', all_slices_similarity)
    
    
# annotator = 0
# image_ids = []
# for i in range(2):
#     for j in range(10):
#         image_ids.append(str(i)+str(j))
# root_paths = [#"/mnt/nas/share2/home/liuke/edata/DPersona1_NPC_20250114-232424results",
#               #"/mnt/nas/share2/home/liuke/edata/DPersona2_NPC_20250115-050904results",
#               #"/mnt/nas/share2/home/liuke/edata/cm_global01_2_NPC_20250122-103651results",
#               #"/mnt/nas/share2/home/liuke/edata/cm_pixel01_2_NPC_20250122-122047results",
#               #"/mnt/nas/share2/home/liuke/edata/prob_unet01_2_NPC_20250122-065829results",
#               #"/mnt/nas/share2/home/liuke/edata/TAB-232522results",
#               #"/mnt/nas/share2/home/liuke/edata/pionono01_NPC_20250115-001034results",
#               "/mnt/nas/share2/home/liuke/edata/pionono_prob01_2_NPC_20250118-012413results"
#               ]
# method_names = [#"DPersona1",
# #               "DPersona2",
# #               "cm_global01_2",
# #               "cm_pixel01_2",
# #               "prob_unet01_2",
# #               "TAB",
# #               "pionono01",
#               "pionono_prob"
#               ]

# for root_path, method_name in zip(root_paths, method_names):
#     print(f'getting distribution of {method_name}')
#     get_distribution(root_path, method_name)
# if method_name not in root_path:
#     raise Exception("method_name not in root_path")



In [ ]:

import numpy as np
from tqdm import tqdm
import nibabel as nib
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np
from lib.metrics_set import  distance_one as distance
import random

def get_data(path):
    nii_file = nib.load(path)
    data = nii_file.get_fdata()
    #print(data.shape)
    if len(data.shape) == 4:
        data = data.reshape(-1, 128, 128)
    return data

def compute_similarity(slice_id, image_data_1, image_data_2, image_data_3, image_data_4):
    slice_1 = image_data_1[slice_id].astype(np.int32)
    slice_2 = image_data_2[slice_id].astype(np.int32)
    slice_3 = image_data_3[slice_id].astype(np.int32)
    slice_4 = image_data_4[slice_id].astype(np.int32)
    #temo_list = [slice_1, slice_2, slice_3, slice_4]
    #random.shuffle(temo_list)
    #slice_1, slice_2, slice_3, slice_4 = temo_list
    #np.random.shuffle([slice_1, slice_2, slice_3, slice_4])
    #print(slice_1[64,64], slice_2[64,64], slice_3[64,64], slice_4[64,64])
    distance_1_3 = distance(slice_1.flatten(), slice_3.flatten())
    distance_1_2 = distance(slice_1.flatten(), slice_2.flatten())
    distance_1_4 = distance(slice_1.flatten(), slice_4.flatten())
    
    distance_2_3 = distance(slice_2.flatten(), slice_3.flatten())
    distance_2_4 = distance(slice_2.flatten(), slice_4.flatten())
    
    distance_3_4 = distance(slice_3.flatten(), slice_4.flatten())
    
    distance_matrix = np.zeros((4,3))
    distance_matrix[0,0] = distance_1_2
    distance_matrix[0,1] = distance_1_3
    distance_matrix[0,2] = distance_1_4
    distance_matrix[1,0] = distance_1_2
    distance_matrix[1,1] = distance_2_3
    distance_matrix[1,2] = distance_2_4
    distance_matrix[2,0] = distance_1_3
    distance_matrix[2,1] = distance_2_3
    distance_matrix[2,2] = distance_3_4
    distance_matrix[3,0] = distance_1_4
    distance_matrix[3,1] = distance_2_4
    distance_matrix[3,2] = distance_3_4
    return distance_matrix
    #return np.array([distance_1_3+distance_1_2, distance_4_3+distance_4_2])

def get_label_distribution(root_path, method_name):
    all_slices_similarity = []
    for image_id in tqdm(image_ids):
        image_path_1 = f'{root_path}/{image_id}_pred_s1.nii.gz'
        image_path_2 = f'{root_path}/{image_id}_pred_s2.nii.gz'
        image_path_3 = f'{root_path}/{image_id}_pred_s3.nii.gz'
        image_path_4 = f'{root_path}/{image_id}_pred_s4.nii.gz'
        image_data_1 = get_data(image_path_1)
        image_data_2 = get_data(image_path_2)
        image_data_3 = get_data(image_path_3)
        image_data_4 = get_data(image_path_4)
        num_slices = image_data_1.shape[0]
        temp_similar_matrices = Parallel(n_jobs=-1)(
            delayed(compute_similarity)(one_slice_id, image_data_1, image_data_2, image_data_3, image_data_4) for one_slice_id in tqdm(range(num_slices))
        )
        all_slices_similarity.append(temp_similar_matrices)
    all_slices_similarity = np.array(all_slices_similarity, dtype=object)
    #print(all_slices_similarity.shape)
    
    np.save(f'{method_name}_rest.npy', all_slices_similarity)

def get_distribution(root_path, method_name):
    all_slices_similarity = []
    for image_id in tqdm(image_ids):
        image_path_1 = f'{root_path}/{image_id}_pred_s1.nii.gz'
        image_path_2 = f'{root_path}/{image_id}_pred_s2.nii.gz'
        image_path_3 = f'{root_path}/{image_id}_pred_s3.nii.gz'
        image_path_4 = f'{root_path}/{image_id}_pred_s4.nii.gz'
        image_data_1 = get_data(image_path_1)
        image_data_2 = get_data(image_path_2)
        image_data_3 = get_data(image_path_3)
        image_data_4 = get_data(image_path_4)
        num_slices = image_data_1.shape[0]
        temp_similar_matrices = Parallel(n_jobs=-1)(
            delayed(compute_similarity)(one_slice_id, image_data_1, image_data_2, image_data_3, image_data_4) for one_slice_id in tqdm(range(num_slices))
        )
        all_slices_similarity.append(temp_similar_matrices)
    all_slices_similarity = np.array(all_slices_similarity, dtype=object)
    #print(all_slices_similarity.shape)
    
    np.save(f'{method_name}_rest.npy', all_slices_similarity)
    
annotator = 0
image_ids = []
for i in range(2):
    for j in range(10):
        image_ids.append(str(i)+str(j))
root_paths = [
            "/mnt/nas/share2/home/liuke/edata/DPersona1_NPC_20250114-232424results",
              "/mnt/nas/share2/home/liuke/edata/DPersona2_NPC_20250115-050904results",
              "/mnt/nas/share2/home/liuke/edata/cm_global01_2_NPC_20250122-103651results",
              "/mnt/nas/share2/home/liuke/edata/cm_pixel01_2_NPC_20250122-122047results",
              "/mnt/nas/share2/home/liuke/edata/prob_unet01_2_NPC_20250122-065829results",
              "/mnt/nas/share2/home/liuke/edata/TAB-232522results",
              "/mnt/nas/share2/home/liuke/edata/pionono01_NPC_20250115-001034results",
              "/mnt/nas/share2/home/liuke/edata/pionono_prob01_2_NPC_20250118-012413results",
              "/data/D-Persona/models/pionono_prob01_2_NPC_20250118-012413results_prior"
              ]
method_names = ["DPersona1",
              "DPersona2",
              "cm_global01_2",
              "cm_pixel01_2",
              "prob_unet01_2",
              "TAB",
              "pionono01",
              "pionono_prob",
              "pionono_prob_prior"
              ]

for root_path, method_name in zip(root_paths, method_names):
    print(f'getting distribution of {method_name}')
    get_distribution(root_path, method_name)
if method_name not in root_path:
    raise Exception("method_name not in root_path")



In [2]:
def get_label_distribution(root_path, method_name):
    all_slices_similarity = []
    for image_id in tqdm(image_ids):
        image_path_1 = f'{root_path}/{image_id}_label_a1.nii.gz'
        image_path_2 = f'{root_path}/{image_id}_label_a2.nii.gz'
        image_path_3 = f'{root_path}/{image_id}_label_a3.nii.gz'
        image_path_4 = f'{root_path}/{image_id}_label_a4.nii.gz'
        image_data_1 = get_data(image_path_1)
        image_data_2 = get_data(image_path_2)
        image_data_3 = get_data(image_path_3)
        image_data_4 = get_data(image_path_4)
        num_slices = image_data_1.shape[0]
        temp_similar_matrices = Parallel(n_jobs=-1)(
            delayed(compute_similarity)(one_slice_id, image_data_1, image_data_2, image_data_3, image_data_4) for one_slice_id in tqdm(range(num_slices))
        )
        all_slices_similarity.append(temp_similar_matrices)
    all_slices_similarity = np.array(all_slices_similarity, dtype=object)
    #print(all_slices_similarity.shape)
    
    np.save(f'vis/sim/Gold_rest.npy', all_slices_similarity)
get_label_distribution("/mnt/nas/share2/home/liuke/prj/D-Persona-main/D-Persona/models/pionono_prob01_2_NPC_20250118-012413results_prior", "pionono_prob_prior")

100%|██████████| 21/21 [00:00<00:00, 24039.41it/s]
/tmp/ipykernel_3294603/2023806768.py:39: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_3294603/2023806768.py:40: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_3294603/2023806768.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
/tmp/ipykernel_3294603/2023806768.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you e